<div align="center">

# Quantum Computing Hackathon &mdash; Track 2

### Qiskit Fall Fest / Quantum Hackathon
**RIFT Labs (Research in Future Technologies)**
**The Catholic University of America, College of Engineering, Physics, and Computing**

</div>

---

In [ ]:
# =====================================================================
#  SETUP  (run this first, it is the only cell that needs the internet)
# =====================================================================
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install pkg only if it is not already importable."""
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    return importlib.import_module(name)

ensure("qiskit")
ensure("numpy")
ensure("scipy")

import numpy as np
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import (Statevector, DensityMatrix, Operator,
                                 SparsePauliOp, Pauli, partial_trace,
                                 state_fidelity)

import itertools, math, time
from fractions import Fraction
from scipy.optimize import minimize
import scipy

print("qiskit", qiskit.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)

# ----- tiny toolkit used throughout (no plotting libraries required) ---
rng = np.random.default_rng(7)
_SCORE = []


def check(label, ok, detail=""):
    """Record and print a pass/fail line. Never raises, so a wrong answer
    does not stop the notebook."""
    _SCORE.append((label, bool(ok)))
    print(("[ PASS ] " if ok else "[ FAIL ] ") + label + (("   " + str(detail)) if detail else ""))
    return bool(ok)


def scoreboard():
    """Summarise every check() run so far."""
    if not _SCORE:
        print("no checks run yet")
        return
    passed = sum(1 for _, ok in _SCORE if ok)
    print("=" * 58)
    print(f"  {passed} / {len(_SCORE)} checks passed")
    print("=" * 58)
    for label, ok in _SCORE:
        print(("  ok   " if ok else "  FAIL ") + label)
    if passed != len(_SCORE):
        raise AssertionError(f"{len(_SCORE) - passed} check(s) failed")


def bar(counts, width=40, title=None, top=None):
    """ASCII histogram of a counts dict. Works in any environment."""
    if title:
        print(title)
    if not counts:
        print("  (no data)")
        return
    items = sorted(counts.items(), key=lambda kv: -kv[1])
    if top:
        items = items[:top]
    total = sum(counts.values())
    hi = max(counts.values())
    for k, v in sorted(items):
        n = int(round(width * v / hi))
        print(f"  |{k}>  {'#' * n:<{width}} {v:>6}  {100 * v / total:5.1f}%")


def sample(circ, shots=2048):
    """Shot-based measurement without needing a backend at all: build the exact
    statevector, then draw multinomial samples from it. Any final measure()
    instructions are stripped first, so you can pass either form."""
    clean = circ.remove_final_measurements(inplace=False)
    sv = Statevector.from_instruction(clean)
    return {str(k): int(v) for k, v in sv.sample_counts(shots).items()}


def bloch(circ):
    """Return (<X>, <Y>, <Z>) for a single-qubit circuit."""
    sv = Statevector.from_instruction(circ)
    return tuple(float(np.real(sv.expectation_value(Pauli(p)))) for p in "XYZ")


def pretty(sv, tol=1e-9):
    """Readable ket expansion of a Statevector."""
    n = sv.num_qubits
    terms = []
    for i, a in enumerate(sv.data):
        if abs(a) > tol:
            re, im = np.real(a), np.imag(a)
            c = f"{re:+.4f}" if abs(im) < tol else f"({re:+.4f}{im:+.4f}j)"
            terms.append(f"{c}|{format(i, '0' + str(n) + 'b')}>")
    return "  ".join(terms) if terms else "0"

---
# Problem 1. VQE: find the ground state energy of H$_2$

**Difficulty: hard**

### The physics

The electronic structure problem is to find the lowest eigenvalue of a molecular
Hamiltonian. For H$_2$ in a minimal basis, after second quantisation and a parity
mapping that exploits the molecule's symmetries, the whole problem collapses onto
**two qubits**:

$$H = g_0 II + g_1 ZI + g_2 IZ + g_3 ZZ + g_4 XX + g_5 YY.$$

The six real coefficients depend on the bond length. We use a set corresponding to
H$_2$ near its equilibrium separation (about 0.735 angstroms), in Hartrees:

$$g_0 = -0.4804,\; g_1 = +0.3435,\; g_2 = -0.3435,\; g_3 = +0.5716,\;
  g_4 = g_5 = +0.0910.$$

Since this is only a $4 \times 4$ matrix we can diagonalise it exactly, which is
precisely why it is a good benchmark: you get to check your variational answer against
the truth. For any molecule big enough to matter, that check is unavailable, which is
the entire motivation for the field.

### The variational principle

For **any** normalised trial state $|\psi(\theta)\rangle$,

$$E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle \;\ge\; E_0.$$

So minimising $E(\theta)$ over the parameters can never take you below the true ground
energy, and the best you reach is an upper bound. The quantum computer's job is only to
evaluate $E(\theta)$; a classical optimiser does the searching. That split is what makes
VQE plausible on noisy hardware.

### Measuring an expectation value for real

On hardware you cannot ask for $\langle H \rangle$ directly. You measure one Pauli term
at a time:

$$\langle H \rangle = \sum_k c_k \langle P_k \rangle,$$

and to measure $\langle P_k \rangle$ you first rotate $P_k$ into the computational basis:
apply $H$ to each qubit where $P_k$ has an $X$, apply $S^\dagger$ then $H$ where it has a
$Y$, leave $Z$ and $I$ alone. Then measure all qubits and take

$$\langle P_k \rangle \approx \frac{1}{\text{shots}} \sum_{\text{shots}} (-1)^{\text{parity of the bits on } \mathrm{supp}(P_k)}.$$

Each term needs its own shot budget, and the variance of each estimate falls as
$1/\text{shots}$. Counting shots properly is most of what makes real VQE expensive.

### Gradients: the parameter-shift rule

Finite differences are a bad idea under shot noise, because you are subtracting two
noisy numbers that are nearly equal. There is an exact alternative. If a parameter
$\theta_i$ enters only through gates of the form $e^{-i\theta_i P / 2}$ with $P$ a Pauli
(so $R_x$, $R_y$, $R_z$ all qualify), then

$$\frac{\partial E}{\partial \theta_i}
 = \frac{E(\theta_i + \tfrac{\pi}{2}) - E(\theta_i - \tfrac{\pi}{2})}{2}.$$

This is **exact**, not an approximation, and the shifts are large so the subtraction is
numerically well conditioned. It holds because such an $E$ is a pure sinusoid in each
parameter individually.

### Your task

Four parts.

- **(a)** Build `H` as a `SparsePauliOp` and get $E_0$ by exact diagonalisation.
- **(b)** Write a hardware-efficient ansatz and an exact energy function, then minimise.
- **(c)** Write `expectation_shots(circ, op, shots)` doing the real basis-rotation
  measurement, and show it converges to the exact value as $1/\sqrt{\text{shots}}$.
- **(d)** Write `grad_parameter_shift` and confirm it matches finite differences.

In [ ]:
G = dict(g0=-0.4804, g1=+0.3435, g2=-0.3435, g3=+0.5716, g4=+0.0910, g5=+0.0910)

def h2_hamiltonian():
    """Return the 2-qubit H2 Hamiltonian as a SparsePauliOp."""
    # YOUR CODE HERE
    return None

def ansatz(theta):
    """Hardware-efficient ansatz. Choose your own structure and parameter count."""
    qc = QuantumCircuit(2)
    # YOUR CODE HERE
    return qc

def energy_exact(theta, H):
    """<psi(theta)|H|psi(theta)> computed exactly from the statevector."""
    # YOUR CODE HERE
    return 0.0

def expectation_shots(circ, op, shots=4096, rng=None):
    """Estimate <op> by basis-rotating each Pauli term and sampling."""
    # YOUR CODE HERE
    return 0.0

def grad_parameter_shift(theta, H):
    """Exact gradient of energy_exact via the parameter-shift rule."""
    # YOUR CODE HERE
    return np.zeros_like(np.asarray(theta, float))

In [ ]:
# ---- (a) exact reference ---------------------------------------------
H = h2_hamiltonian()
spectrum = np.linalg.eigvalsh(H.to_matrix())
E0 = float(spectrum[0])
print("full spectrum (Hartree):", np.round(spectrum, 6))
print(f"exact ground state energy E0 = {E0:.10f} Ha")
check("A1a  Hamiltonian is Hermitian with 4 real eigenvalues",
      np.allclose(H.to_matrix(), H.to_matrix().conj().T))

# ---- (b) optimise both ansaetze --------------------------------------
res_chem = minimize(lambda t: energy_exact(t, H, ansatz_chem), [0.1], method="BFGS")
print(f"\nchemistry ansatz  (1 param):  E = {res_chem.fun:.10f}   "
      f"error = {abs(res_chem.fun - E0):.2e}   theta = {res_chem.x[0]:+.5f}")

best, best_x, hits = np.inf, None, 0
for seed in range(12):
    x0 = np.random.default_rng(seed).uniform(-np.pi, np.pi, 6)
    r = minimize(lambda t: energy_exact(t, H, ansatz_hw), x0, method="BFGS")
    if r.fun < best:
        best, best_x = r.fun, r.x
    if abs(r.fun - E0) < 1e-6:
        hits += 1
print(f"hardware ansatz   (6 param):  E = {best:.10f}   error = {abs(best - E0):.2e}")
print(f"                              {hits}/12 random restarts reached the ground state")

check("A1b  chemistry ansatz reaches the exact ground state",
      abs(res_chem.fun - E0) < 1e-8, f"error {abs(res_chem.fun - E0):.2e}")
check("A1b  hardware-efficient ansatz also reaches it, given restarts",
      abs(best - E0) < 1e-8, f"error {abs(best - E0):.2e}")
check("A1b  the variational bound is never violated",
      res_chem.fun >= E0 - 1e-12 and best >= E0 - 1e-12)
print(f"\nNote: {12 - hits} of 12 restarts got stuck in a local minimum with the richer")
print("ansatz. More parameters is not automatically better.")

In [ ]:
# ---- (c) shot-based measurement, and its noise -----------------------
theta_opt = res_chem.x
rng_s = np.random.default_rng(2024)
print("Estimating <H> by basis rotation and sampling (exact value "
      f"{E0:.6f}):\n")
print("    shots     mean est.      std of est.    1/sqrt(shots)")
ok_scale = True
prev = None
for shots in [64, 256, 1024, 4096, 16384]:
    ests = [expectation_shots(ansatz_chem(theta_opt), H, shots=shots, rng=rng_s)
            for _ in range(60)]
    m, s = float(np.mean(ests)), float(np.std(ests))
    print(f"  {shots:7d}   {m:+.6f}      {s:.6f}       {1 / np.sqrt(shots):.6f}")
    if prev is not None:
        ok_scale &= s < prev * 0.85          # 4x the shots should roughly halve it
    prev = s

check("A1c  the shot estimator is unbiased (converges to the exact energy)",
      abs(np.mean([expectation_shots(ansatz_chem(theta_opt), H, shots=200000, rng=rng_s)
                   for _ in range(3)]) - E0) < 3e-3)
check("A1c  the standard error shrinks like 1/sqrt(shots)", ok_scale)
print("\nThat 1/sqrt(shots) wall is why chemical accuracy (about 1.6 milli-Hartree)")
print("needs millions of shots per energy evaluation on real hardware.")

In [ ]:
# ---- (d) parameter-shift gradient ------------------------------------
t0 = np.array([0.3, -1.1, 0.7, 2.0, -0.4, 1.3])

ps = grad_parameter_shift(t0, H, ansatz_hw)
eps = 1e-6
fd = np.array([(energy_exact(t0 + eps * np.eye(6)[i], H, ansatz_hw)
                - energy_exact(t0 - eps * np.eye(6)[i], H, ansatz_hw)) / (2 * eps)
               for i in range(6)])
print("parameter shift :", np.round(ps, 9))
print("finite difference:", np.round(fd, 9))
print("max abs deviation:", f"{np.max(np.abs(ps - fd)):.2e}")
check("A1d  parameter-shift gradient matches finite differences",
      np.max(np.abs(ps - fd)) < 1e-6)

# use the analytic gradient in the optimiser
r = minimize(lambda t: energy_exact(t, H, ansatz_hw), t0,
             jac=lambda t: grad_parameter_shift(t, H, ansatz_hw), method="BFGS")
print(f"\nBFGS with the analytic gradient: E = {r.fun:.10f} in {r.nit} iterations")
check("A1d  gradient-based optimisation converges", r.fun < E0 + 1e-3)

# Why the rule is exact: E is a pure sinusoid in each parameter separately.
# Fit a + b sin(x) + c cos(x) by least squares and look at the residual.
scan = np.linspace(-np.pi, 3 * np.pi, 400)
vals = np.array([energy_exact(np.r_[t, t0[1:]], H, ansatz_hw) for t in scan])
basis = np.c_[np.ones_like(scan), np.sin(scan), np.cos(scan)]
coef, *_ = np.linalg.lstsq(basis, vals, rcond=None)
resid = float(np.max(np.abs(basis @ coef - vals)))
print(f"\nscanning theta_0 over two full periods and fitting a + b sin + c cos:")
print(f"  fitted coefficients: a = {coef[0]:+.6f}, b = {coef[1]:+.6f}, c = {coef[2]:+.6f}")
print(f"  largest residual over 400 points: {resid:.2e}")
check("A1d  E is a pure sinusoid in each parameter, which is why the rule is exact",
      resid < 1e-9, f"max residual {resid:.2e}")
print("\nThat is the whole justification for parameter shift: a function of the form")
print("a + b sin(x) + c cos(x) has derivative exactly (f(x+pi/2) - f(x-pi/2))/2.")

---
# Problem 2. QAOA for MaxCut on a frustrated graph

**Difficulty: hard**

### The problem

MaxCut: split the vertices of a graph into two sets so that as many edges as possible
cross between them. It is NP-hard. The best known classical approximation guarantee is
the Goemans-Williamson semidefinite programming bound of about **0.878** of optimal, and
beating that is hard under reasonable complexity assumptions.

We use the **triangular prism**: two disjoint triangles $\{0,1,2\}$ and $\{3,4,5\}$,
joined by three rungs $0\!-\!3$, $1\!-\!4$, $2\!-\!5$. Nine edges, six vertices. The
important feature is that it is **not bipartite**: a triangle cannot have all three of
its edges cut, so the graph is *frustrated* and the optimum is strictly less than nine.
That makes the approximation ratio a meaningful number rather than a formality, which it
would not be on a bipartite graph where a perfect cut exists.

### Encoding it as a Hamiltonian

Assign each vertex a spin $z_v = \pm 1$. An edge $(u,v)$ is cut exactly when
$z_u z_v = -1$, so the number of cut edges is

$$C(z) = \sum_{(u,v) \in E} \frac{1 - z_u z_v}{2}.$$

Promote each $z_v$ to the Pauli $Z_v$ and you get a diagonal Hamiltonian whose
eigenvalues on computational basis states are exactly the cut values. Maximising the cut
is finding the **largest** eigenvalue, so we define

$$H_C = \sum_{(u,v)\in E} \frac{Z_u Z_v - 1}{2}$$

whose smallest eigenvalue is $-\mathrm{maxcut}$, and then minimise.

### The QAOA ansatz

Alternate two Hamiltonians for $p$ rounds, starting from the uniform superposition
$|+\rangle^{\otimes n}$:

$$|\psi(\gamma,\beta)\rangle = \prod_{l=1}^{p}
  e^{-i\beta_l H_B}\, e^{-i\gamma_l H_C}\; |+\rangle^{\otimes n},
\qquad H_B = \sum_v X_v.$$

The cost layer $e^{-i\gamma H_C}$ stamps a phase proportional to each candidate's cut
value, and the mixer $e^{-i\beta H_B}$ moves amplitude between candidates. As
$p \to \infty$ this can approach the exact optimum (it contains a Trotterised adiabatic
path as a special case). At finite $p$ it is a heuristic, and measuring how the
approximation ratio climbs with $p$ is the whole experiment.

Implementing $e^{-i\gamma Z_u Z_v}$: conjugate a $Z$ rotation by CNOTs, that is
`cx(u,v); rz(2*gamma, v); cx(u,v)`. The factor of 2 is the usual Qiskit convention,
since `rz(phi)` implements $e^{-i\phi Z/2}$.

### Your task

- **(a)** Brute force the exact MaxCut over all $2^6$ assignments.
- **(b)** Build $H_C$ and verify its eigenvalue on an optimal assignment is $-\mathrm{maxcut}$.
- **(c)** Build the QAOA circuit and an exact $\langle C\rangle$.
- **(d)** Optimise for $p = 1, 2, 3$ and report approximation ratios.
- **(e)** Map the $p=1$ landscape over $(\gamma, \beta)$ and find its structure.

In [ ]:
EDGES = [(0, 1), (1, 2), (2, 0),          # triangle A
         (3, 4), (4, 5), (5, 3),          # triangle B
         (0, 3), (1, 4), (2, 5)]          # the rungs
NV = 6

def cut_value(bits):
    """Number of edges cut by an assignment given as a 0/1 sequence indexed by vertex."""
    return sum(1 for u, v in EDGES if bits[u] != bits[v])

# --- provided for you: the ONE place bit ordering is handled -----------
def bits_of(bitstring):
    """Qiskit bitstring -> list indexed by vertex."""
    return [int(bitstring[NV - 1 - v]) for v in range(NV)]

def to_bitstring(bits):
    """List indexed by vertex -> Qiskit bitstring."""
    return "".join(str(bits[NV - 1 - i]) for i in range(NV))

def brute_force_maxcut():
    """Return (best_cut, list_of_optimal_bitstrings_in_qiskit_order)."""
    # YOUR CODE HERE
    return 0, []

def cost_hamiltonian():
    """H_C = sum over edges of (Z_u Z_v - 1)/2, as a SparsePauliOp on NV qubits."""
    # YOUR CODE HERE
    return None

def qaoa_circuit(params, p):
    """params = [gamma_1..gamma_p, beta_1..beta_p]."""
    qc = QuantumCircuit(NV)
    # YOUR CODE HERE
    return qc

def expected_cut(params, p, HC):
    """Exact <C> = -<H_C> for the QAOA state."""
    # YOUR CODE HERE
    return 0.0

In [ ]:
# ---- (a) and (b) -----------------------------------------------------
maxcut, winners = brute_force_maxcut()
HC = cost_hamiltonian()
print(f"graph: {NV} vertices, {len(EDGES)} edges, not bipartite")
print(f"brute-force MaxCut = {maxcut} of {len(EDGES)} edges")
print(f"{len(winners)} optimal assignments: {winners}")
print("\nFrustration check: each triangle can contribute at most 2 cut edges, so the")
print(f"ceiling is 2 + 2 + 3 = 7, and the optimum reaches it.")

check("A2a  MaxCut is 7, strictly below the edge count, so the graph is frustrated",
      maxcut == 7 and maxcut < len(EDGES))

ev = np.linalg.eigvalsh(HC.to_matrix())
check("A2b  the lowest eigenvalue of H_C is -maxcut",
      abs(ev[0] + maxcut) < 1e-10, f"lambda_min = {ev[0]:.6f}")
for w in winners[:2]:
    e = float(np.real(Statevector.from_label(w).expectation_value(HC)))
    check(f"A2b  <H_C> on optimal assignment {w} equals -{maxcut}", abs(e + maxcut) < 1e-10)

In [ ]:
# ---- (c) and (d): the depth scaling experiment -----------------------
best_params = {}
print("   p   <C> at optimum   ratio <C>/maxcut   P(exact optimum)   gates")
for p in [1, 2, 3]:
    best, bx = -np.inf, None
    for seed in range(8):
        rg = np.random.default_rng(seed)
        x0 = np.concatenate([rg.uniform(0, np.pi, p), rg.uniform(0, np.pi / 2, p)])
        r = minimize(lambda x: -expected_cut(x, p, HC), x0,
                     method="COBYLA", options={"maxiter": 1500})
        if -r.fun > best:
            best, bx = -r.fun, r.x
    best_params[p] = bx
    probs = Statevector(qaoa_circuit(bx, p)).probabilities_dict()
    p_opt = sum(pr for b, pr in probs.items() if cut_value(bits_of(b)) == maxcut)
    ng = sum(qaoa_circuit(bx, p).count_ops().values())
    print(f"  {p}    {best:9.5f}        {best / maxcut:.5f}          {p_opt:.5f}        {ng:4d}")

ratios = [expected_cut(best_params[p], p, HC) / maxcut for p in [1, 2, 3]]
check("A2d  the approximation ratio improves monotonically with depth p",
      ratios[0] < ratios[1] < ratios[2],
      f"{ratios[0]:.4f} -> {ratios[1]:.4f} -> {ratios[2]:.4f}")
check("A2d  p=3 QAOA beats the Goemans-Williamson 0.878 guarantee on this instance",
      ratios[2] > 0.878, f"ratio = {ratios[2]:.4f}")

p = 3
probs = Statevector(qaoa_circuit(best_params[3], 3)).probabilities_dict()
top = sorted(probs.items(), key=lambda kv: -kv[1])[:6]
print("\nMost likely measurement outcomes at p=3:")
for b, pr in top:
    print(f"  {b}   P = {pr:.4f}   cut = {cut_value(bits_of(b))}"
          + ("   <-- optimal" if cut_value(bits_of(b)) == maxcut else ""))
print("\nNote that <C> being 0.9x optimal does not mean you sample the optimum 90% of")
print("the time. You sample good-but-not-best cuts a lot, which is why in practice you")
print("take many shots and keep the best one you see.")

In [ ]:
# ---- (e) the p=1 landscape -------------------------------------------
res = 41
gs = np.linspace(0, np.pi, res)
bs = np.linspace(0, np.pi, res)
land = np.array([[expected_cut([g, b], 1, HC) for g in gs] for b in bs])

i, j = np.unravel_index(np.argmax(land), land.shape)
print(f"grid best at gamma = {gs[j]:.4f}, beta = {bs[i]:.4f},  <C> = {land[i, j]:.5f}")
print(f"optimiser found      gamma = {best_params[1][0] % np.pi:.4f}, "
      f"beta = {best_params[1][1] % np.pi:.4f}, <C> = {expected_cut(best_params[1], 1, HC):.5f}")

print("\n<C> over the (gamma, beta) square, coarse view (rows = beta, cols = gamma):")
lo, hi = land.min(), land.max()
chars = " .:-=+*#%@"
for r_i in range(0, res, 2):
    row = "".join(chars[min(len(chars) - 1, int((land[r_i, c] - lo) / (hi - lo + 1e-12) * len(chars)))]
                  for c in range(0, res, 1))
    print(f"  beta={bs[r_i]:4.2f} |{row}|")
print(f"         gamma: {gs[0]:.2f} {'-' * (res - 14)} {gs[-1]:.2f}")

opt1 = expected_cut(best_params[1], 1, HC)
gap = opt1 - land[i, j]
print(f"\ngrid spacing is {gs[1] - gs[0]:.4f} rad, and the grid best falls short of the")
print(f"optimiser by {gap:.5f}, which is what you expect from discretisation alone.")
check("A2e  the grid never beats the optimiser, and lands close to it",
      -1e-9 <= gap < 0.05, f"optimiser {opt1:.5f} vs grid {land[i, j]:.5f}")
check("A2e  the landscape is periodic in beta with period pi/2",
      np.allclose(np.array([expected_cut([0.7, b], 1, HC) for b in [0.3, 0.3 + np.pi / 2]]),
                  expected_cut([0.7, 0.3], 1, HC), atol=1e-9))
print("\nThe landscape has structure: it is smooth, periodic, and has only a handful of")
print("basins. That is why QAOA parameters transfer between similar instances, and why")
print("people bother fitting parameter schedules instead of optimising from scratch.")

---
# Problem 3. Order finding, and factoring a number with it

**Difficulty: very hard**

### What Shor's algorithm actually is

Factoring is reduced to **order finding**, and only the order finding is quantum. Given
$N$ and an $a$ coprime to it, the order $r$ is the smallest positive integer with

$$a^{r} \equiv 1 \pmod N.$$

Once you know $r$, and if $r$ is even and $a^{r/2} \not\equiv -1 \pmod N$, then
$a^{r/2} - 1$ and $a^{r/2} + 1$ share a nontrivial factor with $N$, which
$\gcd$ hands you in microseconds. Everything expensive is in finding $r$.

Order finding is phase estimation on the modular multiplication operator

$$U_a : |x\rangle \mapsto |a x \bmod N\rangle,$$

whose eigenvalues are $e^{2\pi i s / r}$ for $s = 0, 1, \ldots, r-1$. Run phase
estimation on $U_a$ with the work register started in $|1\rangle$ (which is an equal
superposition of all $r$ relevant eigenvectors) and you measure an estimate of $s/r$ for
a random $s$. Then you recover $r$ classically.

### Two things that will bite you

**Precision.** The counting register needs enough qubits that $s/r$ is resolvable. The
standard requirement is

$$m \;\ge\; 2\lceil \log_2 N \rceil + 1,$$

so that the measured value is within $1/2N^2$ of $s/r$, which is close enough for
continued fractions to be guaranteed to recover the right denominator. Skimp on $m$ and
the algorithm does not degrade gracefully, it just returns garbage. You will see this
below in the starkest possible way: $N = 21$ with $m = 5$ succeeds **zero** percent of
the time, and with $m = 7$ it works. Meanwhile $N = 15$ with $a = 7$ works at $m = 4$,
because there $r = 4$ is a power of two and the phase is exactly representable in four
bits. That is a special case, and treating it as typical is the single most common way
people fool themselves with a toy Shor implementation.

**Recovering $r$ from the measurement.** You measure $y$ and believe
$y/2^m \approx s/r$. Recover $r$ with the continued fraction expansion of $y/2^m$,
taking the best rational approximation with denominator below $N$. Python's
`fractions.Fraction(y, 2**m).limit_denominator(N)` does exactly that. You may get a
divisor of $r$ rather than $r$ itself (notably $s = 0$ gives $r = 1$, useless), so you
verify each candidate with $a^{r} \equiv 1$ and retry on failure. Shor's algorithm is
a Las Vegas algorithm: it either succeeds or admits it, never silently lies, provided
you check.

### Your task

- **(a)** Write `modmul_gate(a, N, n)` building $U_a$ as a genuine unitary on $n$ qubits.
  It must be a permutation matrix. Act as the identity on $x \ge N$, since those basis
  states are outside the modular group and must still be mapped reversibly.
- **(b)** Write `order_finding(a, N, m)` doing the full phase estimation, with controlled
  $U_{a^{2^j}}$ obtained by **repeated squaring of the exponent**, not by applying $U_a$
  a huge number of times.
- **(c)** Write `recover_order` using continued fractions.
- **(d)** Write `shor_factor(N, m)` putting it together, and factor 15 and 21.
- **(e)** Show the precision requirement empirically.

In [ ]:
from qiskit.circuit.library import UnitaryGate, QFTGate

# --- provided for you: the classical reference answer -----------------
def true_order(a, N):
    """Classical order of a mod N, for checking your quantum result."""
    r, v = 1, a % N
    while v != 1:
        v, r = (v * a) % N, r + 1
    return r

def modmul_gate(a, N, n, label=None):
    """Unitary on n qubits implementing |x> -> |a*x mod N> for x < N, identity above."""
    # YOUR CODE HERE: build the permutation matrix, assert unitarity, wrap in UnitaryGate
    return None

def order_finding(a, N, m):
    """Phase estimation on U_a. Returns the probability distribution over the
    m-bit counting register."""
    # YOUR CODE HERE
    return None

def recover_order(y, m, N):
    """Best rational approximation of y/2^m with denominator < N; return the denominator."""
    # YOUR CODE HERE
    return 1

def shor_factor(N, m, attempts=25, seed=0):
    """Full loop. Returns (factor1, factor2, a, r) or None."""
    # YOUR CODE HERE
    return None

In [ ]:
# ---- (a) is the gate right? ------------------------------------------
ok = True
for a, N in [(7, 15), (2, 15), (4, 21), (5, 21), (2, 9)]:
    n = math.ceil(math.log2(N))
    M = Operator(modmul_gate(a, N, n)).data
    ok &= np.allclose(M @ M.conj().T, np.eye(2 ** n), atol=1e-12)
    # check the permutation really is x -> a x mod N
    for x in range(N):
        ok &= abs(M[(a * x) % N, x] - 1) < 1e-12
    r = true_order(a, N)
    # U_a^r must be the identity on the modular subspace
    Mr = np.linalg.matrix_power(M, r)
    ok &= np.allclose(Mr, np.eye(2 ** n), atol=1e-10)
    print(f"  a={a:2d} N={N:2d} n={n}  unitary, correct permutation, U^r = I with r = {r}")
check("A3a  modmul_gate is a correct unitary permutation with U^r = I", ok)

# the eigenvalues really are exp(2 pi i s / r)
a, N = 7, 15
M = Operator(modmul_gate(a, N, 4)).data
phases = np.angle(np.linalg.eigvals(M)) / (2 * np.pi) % 1.0
r = true_order(a, N)
print(f"\nphases of U_{a} mod {N}, as fractions of 2pi: "
      f"{sorted(set(np.round(phases, 6)))}")
print(f"expected multiples of 1/r = 1/{r}: {[s / r for s in range(r)]}")
check("A3a  the eigenphases are multiples of 1/r",
      all(min(abs(ph - s / r) for s in range(r)) < 1e-8 for ph in phases))

In [ ]:
# ---- (b), (c), (e): precision, the thing that actually matters -------
print("Probability that one shot yields the true order r:\n")
print("   a   N    r   m   2log2(N)+1   P(success)")
ok_prec = True
rows = {}
for a, N, ms in [(7, 15, [4, 7]), (4, 21, [5, 7, 9]), (5, 21, [5, 7, 9])]:
    rt = true_order(a, N)
    need = 2 * math.ceil(math.log2(N)) + 1
    for m in ms:
        P = order_finding(a, N, m)
        succ = float(sum(P[y] for y in range(2 ** m) if recover_order(y, m, N) == rt))
        rows[(a, N, m)] = succ
        print(f"  {a:2d}  {N:2d}   {rt}  {m:2d}      {need:2d}        {succ:.4f}")

check("A3e  N=15 with a=7 succeeds even at m=4, because r=4 is a power of two",
      rows[(7, 15, 4)] > 0.4, f"P = {rows[(7, 15, 4)]:.4f}")
check("A3e  N=21 fails completely at m=5, and works at m=7",
      rows[(4, 21, 5)] < 1e-6 < rows[(4, 21, 7)],
      f"P(m=5) = {rows[(4, 21, 5)]:.6f}, P(m=7) = {rows[(4, 21, 7)]:.4f}")
check("A3e  more counting qubits never hurt", rows[(4, 21, 9)] >= rows[(4, 21, 7)] - 1e-9)

print("\nLook at the m=5 row for N=21: not 'degraded', not 'noisy'. Exactly zero.")
print("Under-resourcing the counting register does not give you a worse answer, it")
print("gives you no answer at all. Budget m >= 2*ceil(log2 N) + 1 before you start.")

In [ ]:
# ---- (d) factor things -----------------------------------------------
print("Full Shor, including the classical wrapper:\n")
ok = True
for N, m, seed in [(15, 7, 3), (21, 7, 1), (21, 9, 5), (15, 4, 11)]:
    t = time.time()
    out = shor_factor(N, m, attempts=30, seed=seed)
    el = time.time() - t
    if out is None:
        print(f"  N={N:2d} m={m}: no factor found in the attempt budget")
        ok = False
        continue
    f1, f2, a_used, r_used = out
    good = f1 * f2 == N and 1 < f1 < N
    ok &= good
    via = "gcd shortcut" if r_used is None else f"a={a_used}, r={r_used}"
    print(f"  N={N:2d} m={m}:  {N} = {f1} x {f2}   via {via}   ({el:.1f}s)")

check("A3d  Shor factors both 15 and 21 correctly", ok)

# and show the distribution behind one run
P = order_finding(7, 15, 4)
print("\nCounting-register distribution for a=7, N=15, m=4:")
bar({format(y, "04b"): int(round(P[y] * 10000)) for y in range(16) if P[y] > 1e-9},
    title="", width=34)
print("Four equally likely peaks at y = 0, 4, 8, 12, that is y/16 in {0, 1/4, 1/2, 3/4}.")
print("y=0 gives r=1 and y=8 gives r=2, both useless. Only y=4 and y=12 give r=4.")
print("So one shot succeeds half the time, and you simply retry. That is the")
print("Las Vegas structure: it never lies, it only sometimes shrugs.")

---
# Problem 4. The Steane code, and where error correction starts paying off

**Difficulty: very hard**

### The idea

A $[[n,k,d]]$ stabiliser code encodes $k$ logical qubits into $n$ physical ones and
corrects up to $\lfloor (d-1)/2 \rfloor$ errors. The **Steane code** is $[[7,1,3]]$: seven
physical qubits, one logical qubit, distance 3, so it corrects any single qubit error.

It is a CSS code built from the classical $[7,4,3]$ Hamming code, with six stabiliser
generators. Writing supports as qubit index sets:

$$\begin{aligned}
g_1 &= X_0 X_2 X_4 X_6, & g_4 &= Z_0 Z_2 Z_4 Z_6,\\
g_2 &= X_1 X_2 X_5 X_6, & g_5 &= Z_1 Z_2 Z_5 Z_6,\\
g_3 &= X_3 X_4 X_5 X_6, & g_6 &= Z_3 Z_4 Z_5 Z_6.
\end{aligned}$$

Those supports are the three parity checks of the Hamming code, so qubit $i$ participates
in check $c$ exactly when bit $c$ of $(i+1)$ is set. The code space is the simultaneous
$+1$ eigenspace of all six, which is $2^7 / 2^6 = 2$ dimensional, hence one logical
qubit. The logical operators are $X_L = X^{\otimes 7}$ and $Z_L = Z^{\otimes 7}$.

The $X$ and $Z$ supports being identical makes this code **self-dual**, which is why
$|+_L\rangle = H^{\otimes 7}|0_L\rangle$ and why the code has a transversal Hadamard.
That is a genuinely useful property, not a curiosity: transversal gates are the cheap
ones in a fault-tolerant architecture.

### Encoding

$|0_L\rangle$ is the uniform superposition over the eight codewords in the span of the
three $X$ supports:

$$|0_L\rangle = \frac{1}{\sqrt 8}\sum_{v \in \mathrm{span}\{g_1,g_2,g_3\}} |v\rangle.$$

So the encoder is three Hadamards on "pivot" qubits (one that appears in only one
generator) and nine CNOTs fanning each generator out. Qubit 0 appears only in $g_1$,
qubit 1 only in $g_2$, qubit 3 only in $g_3$, which hands you the pivots.

### Syndromes, and why we can be exact here

Here is the key structural fact that makes this problem tractable. Suppose the state is
$E|\psi_L\rangle$ for some Pauli error $E$. Then for each stabiliser $S$,

$$\langle \psi_L | E^\dagger S E | \psi_L \rangle = \pm 1,$$

with the sign $+1$ if $E$ commutes with $S$ and $-1$ if it anticommutes. So the syndrome
is **deterministic** and depends only on the commutation pattern, which is pure bit
arithmetic: writing $E$ as masks $(e_x, e_z)$ and $S$ as $(s_x, s_z)$, they anticommute
exactly when

$$|e_x \wedge s_z| + |e_z \wedge s_x| \;\text{is odd}.$$

That means we do not need to simulate noise channels on density matrices at all. We can
**enumerate all $4^7 = 16384$ Pauli error patterns**, compute each one's probability
under depolarising noise, decode it, and classify the result. The logical error rate
comes out exact, as a number not an estimate, with no Monte Carlo error bars.

Classifying the outcome is the last trick. After correction the net Pauli $E \cdot C$ has
zero syndrome, so it lies in the stabiliser group $\mathcal S$ or in one of the cosets
$\mathcal S X_L$, $\mathcal S Z_L$, $\mathcal S Y_L$. Every stabiliser generator has even
$X$-weight and even $Z$-weight, while $X_L$ and $Z_L$ have weight 7, odd. Therefore:

> the correction **succeeded** exactly when the net Pauli has even $X$-weight and even
> $Z$-weight.

### The pseudo-threshold

Under depolarising noise at rate $p$ per qubit, the logical error rate of a distance-3
code behaves as $L(p) \approx A p^2$ for small $p$, since it takes two errors to fool the
decoder. An unencoded qubit fails at rate $p$. So:

- for small $p$, $A p^2 < p$ and encoding **helps**,
- for large $p$, $A p^2 > p$ and encoding makes things **worse**, because you now have
  seven chances to fail instead of one.

The crossing point $p^\star = 1/A$ is the **pseudo-threshold**. Finding it is the point
of this problem, and the fact that it exists at all is the reason quantum error
correction is an engineering discipline and not a free win.

### Your task

- **(a)** Build the encoder for $|0_L\rangle$ and verify all six stabilisers give $+1$.
- **(b)** Build $|1_L\rangle$ and $|+_L\rangle$ and verify the logical operators.
- **(c)** Build the syndrome function and the decoder lookup table; confirm all 22
  weight-$\le 1$ Paulis have distinct syndromes.
- **(d)** Compute the exact logical error rate by enumeration.
- **(e)** Find the pseudo-threshold and extract the coefficient $A$.

In [ ]:
# Hamming parity checks: qubit i is in check c iff bit c of (i+1) is set
XSUP = [[0, 2, 4, 6], [1, 2, 5, 6], [3, 4, 5, 6]]
ZSUP = [[0, 2, 4, 6], [1, 2, 5, 6], [3, 4, 5, 6]]

def pauli_on(kind, support, n=7):
    """SparsePauliOp for `kind` acting on `support`, identity elsewhere."""
    return SparsePauliOp.from_sparse_list([(kind * len(support), support, 1.0)],
                                          num_qubits=n)

STABS = [pauli_on("X", s) for s in XSUP] + [pauli_on("Z", s) for s in ZSUP]
X_L = pauli_on("X", list(range(7)))
Z_L = pauli_on("Z", list(range(7)))

def encode_zero():
    """Circuit preparing |0_L> of the Steane code."""
    qc = QuantumCircuit(7)
    # YOUR CODE HERE: h on each pivot, then fan out its generator with CNOTs
    return qc

def syndrome(ex, ez):
    """6-bit syndrome of the Pauli with X-mask ex and Z-mask ez."""
    # YOUR CODE HERE
    return (0,) * 6

def build_decoder():
    """Map syndrome -> (correction_x_mask, correction_z_mask, name)."""
    # YOUR CODE HERE
    return {}

def logical_error_rate(p, table):
    """Exact logical error rate under depolarising noise at rate p per qubit,
    by enumerating all 4**7 Pauli patterns."""
    # YOUR CODE HERE
    return 0.0

In [ ]:
# ---- (a) and (b) -----------------------------------------------------
sv0 = Statevector(encode_zero())
evs = [float(np.real(sv0.expectation_value(s))) for s in STABS]
print("stabiliser expectation values on |0_L>:", np.round(evs, 9))
check("A4a  |0_L> is a +1 eigenstate of all six stabilisers",
      np.allclose(evs, 1.0, atol=1e-10))
check("A4a  |0_L> has <Z_L> = +1",
      abs(float(np.real(sv0.expectation_value(Z_L))) - 1) < 1e-10)

support = [format(i, "07b") for i in range(128) if abs(sv0.data[i]) > 1e-9]
print(f"\n|0_L> is a uniform superposition of {len(support)} codewords:")
print("  " + "  ".join(support))
check("A4a  the superposition has exactly 8 terms, as span{g1,g2,g3} requires",
      len(support) == 8)

c1 = encode_zero(); c1.x(range(7))                 # |1_L> = X_L |0_L>
cp = encode_zero(); cp.h(range(7))                 # |+_L> = H^7 |0_L>, self-duality
sv1, svp = Statevector(c1), Statevector(cp)
check("A4b  |1_L> is still in the code space, with <Z_L> = -1",
      np.allclose([float(np.real(sv1.expectation_value(s))) for s in STABS], 1.0, atol=1e-10)
      and abs(float(np.real(sv1.expectation_value(Z_L))) + 1) < 1e-10)
check("A4b  <0_L|1_L> = 0, so they really are two orthogonal logical states",
      abs(np.vdot(sv0.data, sv1.data)) < 1e-12)
check("A4b  the transversal Hadamard maps |0_L> to |+_L>, confirming self-duality",
      np.allclose([float(np.real(svp.expectation_value(s))) for s in STABS], 1.0, atol=1e-10)
      and abs(float(np.real(svp.expectation_value(X_L))) - 1) < 1e-10)

In [ ]:
# ---- (c) the decoder -------------------------------------------------
table = build_decoder()
print(f"{len(table)} distinct syndromes for the 1 + 21 Paulis of weight <= 1")
check("A4c  all 22 weight-<=1 Paulis have distinct syndromes, so all are correctable",
      len(table) == 22)

print("\nsyndrome table (X-stabs | Z-stabs  ->  correction):")
for synd, (_, _, name) in sorted(table.items(), key=lambda kv: kv[1][2]):
    print(f"  {''.join(map(str, synd[:3]))} | {''.join(map(str, synd[3:]))}   ->  {name}")

# a correctable error really is corrected, verified on the actual statevector
ok = True
for q in range(7):
    for name, op in [("X", lambda c, t: c.x(t)), ("Z", lambda c, t: c.z(t)),
                     ("Y", lambda c, t: c.y(t))]:
        circ = encode_zero()
        op(circ, q)
        ex = (1 << q) if name in ("X", "Y") else 0
        ez = (1 << q) if name in ("Z", "Y") else 0
        cx, cz, _ = table[syndrome(ex, ez)]
        for i in range(7):
            if cx >> i & 1:
                circ.x(i)
            if cz >> i & 1:
                circ.z(i)
        ok &= state_fidelity(Statevector(circ), sv0) > 1 - 1e-10
check("A4c  every single-qubit X, Y or Z error is corrected exactly (21 of 21)", ok)

In [ ]:
# ---- (d) and (e): the exact logical error rate ------------------------
t = time.time()
print("       p      unencoded    Steane [[7,1,3]]    ratio L/p")
pts = [0.30, 0.20, 0.15, 0.10, 0.08, 0.0602, 0.05, 0.03, 0.02, 0.01, 0.005, 0.001]
L = {}
for p in pts:
    L[p] = logical_error_rate(p, table)
    flag = "   <-- encoding helps below here" if abs(p - 0.0602) < 1e-9 else ""
    print(f"  {p:7.4f}   {p:10.6f}    {L[p]:14.8f}    {L[p] / p:7.3f}{flag}")
print(f"\n(exact enumeration of 4^7 = {4 ** 7} patterns per point, {time.time() - t:.1f}s total)")

check("A4d  encoding helps at low noise (p = 0.001)", L[0.001] < 0.001,
      f"L = {L[0.001]:.2e} vs p = 1.0e-03, a {0.001 / L[0.001]:.0f}x improvement")
check("A4d  encoding hurts at high noise (p = 0.20)", L[0.20] > 0.20,
      f"L = {L[0.20]:.4f} vs p = 0.2000")

# bisect for the crossing
lo, hi = 1e-3, 0.4
for _ in range(60):
    mid = 0.5 * (lo + hi)
    if logical_error_rate(mid, table) < mid:
        lo = mid
    else:
        hi = mid
pstar = 0.5 * (lo + hi)
A_coeff = logical_error_rate(1e-4, table) / 1e-8
print(f"\npseudo-threshold   p* = {pstar:.5f}")
print(f"small-p coefficient A = {A_coeff:.4f}   (and 1/A = {1 / A_coeff:.5f})")
print(f"quadratic scaling check: L(1e-3)/L(1e-4) = "
      f"{logical_error_rate(1e-3, table) / logical_error_rate(1e-4, table):.2f}, "
      f"expected about 100 for L ~ p^2")

check("A4e  the pseudo-threshold is near p = 0.06", 0.05 < pstar < 0.07,
      f"p* = {pstar:.5f}")
check("A4e  L(p) ~ A p^2 with A near 21, one per pair of qubits",
      abs(A_coeff - 21) < 1.0, f"A = {A_coeff:.3f}, and C(7,2) = 21")

print(f"\nThe leading-order estimate of the crossing is 1/A = {1 / A_coeff:.5f}, against")
print(f"the true crossing at {pstar:.5f}, so it is about {100 * abs(1 / A_coeff - pstar) / pstar:.0f}% off.")
print("That is not an error in either number. At p ~ 0.06 the cubic and higher terms in")
print("L(p) are no longer negligible, so a purely quadratic model under-predicts the")
print("crossing. Quote p* from the exact calculation, and use 1/A only for intuition.")
check("A4e  1/A predicts the crossing to within 30%, as a leading-order estimate should",
      abs(1 / A_coeff - pstar) / pstar < 0.30,
      f"1/A = {1 / A_coeff:.5f} vs p* = {pstar:.5f}")

print("\nSo: to benefit from this code your physical error rate must already be below")
print(f"about {pstar * 100:.1f}% per qubit. Above that, the seven qubits you added are")
print("seven new ways to fail and the encoding is actively counterproductive. The whole")
print("fault-tolerance programme is the business of getting under thresholds like this")
print("one and then concatenating or scaling the code to push the logical rate down.")

---
# Scoreboard

In [ ]:
scoreboard()

---
### Organisers only

---
<div align="center">
<b>RIFT Labs</b> (Research in Future Technologies)<br>
<b>The Catholic University of America</b>, College of Engineering, Physics, and Computing
</div>